# Supreme Court Case Prioritization & Scheduling - Exploratory Data Analysis
**Academic ML Project: Decision-Support Agent for Judicial Case Prioritization**

This notebook provides comprehensive exploratory data analysis of the Supreme Court dataset (`supreme_court_dataset.csv`), including:
1. Dataset ingestion, schema audit, and missing value analysis
2. Target class distributions (Priority Levels & Case Duration)
3. Categorical distribution of substantive case types
4. Feature correlation and inter-dependency analysis
5. Strict data leakage analysis and pending-case feature boundary definitions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('../data/raw/supreme_court_dataset.csv')
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

## 1. Missing Values & Duplicate Check

In [2]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows count:', df.duplicated().sum())

## 2. Priority Distribution (Target 1 for Model 1)

In [3]:
priority_counts = df['priority_level_simulated'].value_counts()
print(priority_counts)
print('\nPercentages:')
print((priority_counts / len(df) * 100).round(2))

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='priority_level_simulated', palette=['#64748b', '#d97706', '#b91c1c'])
plt.title('Distribution of Case Priority Levels')
plt.xlabel('Priority Tier')
plt.ylabel('Count')
plt.show()

## 3. Case Duration Distribution (Target 2 for Model 2)

In [4]:
print(df['case_duration_days_simulated'].describe())

plt.figure(figsize=(9, 4))
sns.histplot(df['case_duration_days_simulated'], bins=40, kde=True, color='#1e3a8a')
plt.title('Distribution of Case Disposal Duration (Days)')
plt.xlabel('Simulated Duration in Days')
plt.ylabel('Frequency')
plt.show()

## 4. Case Type vs Priority & Duration Analysis

In [5]:
cross_tab = pd.crosstab(df['case_type'], df['priority_level_simulated'], normalize='index') * 100
print('Cross-tabulation (% by row):')
print(cross_tab.round(1))

plt.figure(figsize=(10, 5))
df.groupby('case_type')['case_duration_days_simulated'].mean().sort_values().plot(kind='barh', color='#1e3a8a')
plt.title('Mean Case Duration by Category (Days)')
plt.xlabel('Average Duration (Days)')
plt.show()

## 5. Strict Data Leakage Audit
- `judgment_date`: strictly post-adjudication; MUST NEVER be an input feature.
- `case_duration_days_simulated`: equals `judgment_date - filing_date_simulated`; target for Model 2 ONLY.
- Legitimate prediction features available at pending stage: `case_type`, `case_year`, `num_hearings_simulated`, `bench_size`.